# Independent validation of the saved motor-condition model
Run this in a **new Colab notebook** (not your training notebook). Best practice: **Runtime, Restart session**, then run the cells below top to bottom, so nothing from the training run is still in memory.

It loads the **saved model files** and your **original CSV**, rebuilds the held-out test motors, and checks whether the saved model reproduces the numbers your training notebook recorded. It does **not** train a replacement model, never edits your files, and never invents a number. If a file is missing, the check says **NOT VALIDATED** and tells you why.

**Target A** (Normal / Alert / Danger) is checked. **Target B** (verified failure within seven days) cannot be validated: there is no failure-event data and no model for it.

## Step 1. Install the tools (about 10 seconds)

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib

In [ ]:
%%writefile pm_data_audit.py
#!/usr/bin/env python3
"""pm_data_audit.py - read-only data-quality and leakage audit for the motor condition dataset.

WHAT IT DOES
  Runs independent checks on the RAW master CSV (two header rows, 18 columns) and, optionally, on the
  cleaned working copy and the exported model bundle. It has its OWN loader on purpose: if it reused the
  training notebook's loader, a bug there would be invisible here.

WHAT IT NEVER DOES
  * It never writes to, renames or edits the input file (SHA-256 is recorded before and after).
  * It never fills, drops or invents values or labels.
  * It never says "valid". The strongest verdict is "structure checks passed, label provenance UNVERIFIED",
    because no program can verify who assigned the labels.

USAGE (Colab or any machine)
  python pm_data_audit.py --data motor_predictive_maintenance_master.csv --out audit_outputs
  python pm_data_audit.py --data RAW.csv --clean-copy motor_clean_working_copy.csv --bundle pm_condition_ann_v1.0.0 --out audit_outputs

Exit code: 0 = no FAIL, 1 = at least one FAIL, 2 = could not run.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import re
import sys
from collections import OrderedDict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

AUDIT_VERSION = "1.0.0"

# ----------------------------------------------------------------------------------------------
# Contract copied from the inspection report of the real file (NOT from the notebook loader).
# ----------------------------------------------------------------------------------------------
ID_HEADERS = ["Record ID", "Motor ID", "Measurement timestamps"]
# raw sub-heading -> (clean name used by the notebook/backend, unit)
SENSORS = OrderedDict([
    ("DE_H_mm/s", ("de_h_velocity_mm_s", "mm/s")), ("DE_H_gE", ("de_h_ge", "gE")), ("DE_H_temp_C", ("de_h_temp_c", "degC")),
    ("DE_V_mm/s", ("de_v_velocity_mm_s", "mm/s")), ("DE_V_gE", ("de_v_ge", "gE")), ("DE_V_temp_C", ("de_v_temp_c", "degC")),
    ("NDE_H_mm/s", ("nde_h_velocity_mm_s", "mm/s")), ("NDE_H_gE", ("nde_h_ge", "gE")), ("NDE_H_temp_C", ("nde_h_temp_c", "degC")),
    ("NDE_V_mm/s", ("nde_v_velocity_mm_s", "mm/s")), ("NDE_V_gE", ("nde_v_ge", "gE")), ("NDE_V_temp_C", ("nde_v_temp_c", "degC")),
    ("rotational_speed_rpm", ("rotational_speed_rpm", "rpm")), ("load_percent", ("load_percent", "%")),
])
LABEL_RAW = "condition_label"
CLASSES = ["Normal", "Alert", "Danger"]
SUBHEADINGS = list(SENSORS) + [LABEL_RAW]
CLEAN = [v[0] for v in SENSORS.values()]
FEATURES = CLEAN                                  # the order the backend must use
TS_MDY, TS_DMY = "%m/%d/%Y %H:%M", "%d/%m/%Y %H:%M"

# What the earlier inspection of the real file reported (used only to detect "this is a different file").
INSPECTED = {"sha256": "0270fc74bc9f428b97a0ab8c9cad22fa92a5916edeea98ea9fdf93a795891a8b", "bytes": 216691,
             "rows": 1938, "motors": 40, "labels": {"Normal": 1475, "Alert": 347, "Danger": 116}}

# Generic engineering screens. They only raise WARN for a human to review. They are NOT validity rules,
# and nothing is ever removed because of them.
PLAUSIBLE = {"mm/s": (0.0, 50.0), "gE": (0.0, 100.0), "degC": (-20.0, 200.0), "rpm": (1.0, 4000.0), "%": (0.0, 150.0)}
SENTINELS = {-9999.0, -999.0, -99.0, -9.0, 999.0, 9999.0, 99999.0}
FAIL, WARN, PASS, INFO, SKIP = "FAIL", "WARN", "PASS", "INFO", "SKIP"

# Reproduces notebook v1.0.0 section 11 (split by motors, label counts only). Re-implemented here so the
# audit can check the split WITHOUT importing the notebook. Compare its table with the one the notebook printed.
SEED, SPLIT_FRACTIONS, MIN_CLASS_RECORDS, N_SPLIT_TRIALS = 42, (0.60, 0.20, 0.20), 10, 5000


class Audit:
    def __init__(self):
        self.findings: list[dict] = []
        self.tables: dict[str, object] = {}

    def add(self, fid, title, status, detail, evidence=None):
        self.findings.append({"id": fid, "title": title, "status": status, "detail": detail, "evidence": evidence or {}})
        return status

    def worst(self):
        order = [FAIL, WARN, INFO, SKIP, PASS]
        return min((f["status"] for f in self.findings), key=order.index, default=PASS)


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------------------------
# 1. File and structure
# ----------------------------------------------------------------------------------------------
def read_text_grid(path, a: Audit):
    raw = Path(path).read_bytes()
    bom = raw.startswith(b"\xef\xbb\xbf")
    nul, crlf = raw.count(b"\x00"), raw.count(b"\r\n")
    lf_only = raw.count(b"\n") - crlf
    sha0 = sha256_of(path)
    a.add("F01", "File fingerprint", INFO, f"{len(raw):,} bytes; SHA-256 {sha0}; UTF-8 BOM: {bom}; NUL bytes: {nul}; CRLF lines: {crlf}; LF-only lines: {lf_only}",
          {"sha256": sha0, "bytes": len(raw)})
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        a.add("F02", "Text encoding", FAIL, f"File is not valid UTF-8: {exc}")
        text = raw.decode("latin-1")
    else:
        a.add("F02", "Text encoding", PASS, "Decodes cleanly as UTF-8.")
    sha = sha256_of(path)
    if sha == INSPECTED["sha256"]:
        a.add("F03", "Same file as the earlier inspection", PASS, "SHA-256 equals the file described in the inspection report.")
    else:
        a.add("F03", "Same file as the earlier inspection", WARN,
              "SHA-256 differs from the inspected file. This is a DIFFERENT or EDITED file; every earlier number may be stale. "
              f"inspected={INSPECTED['sha256'][:16]}..., now={sha[:16]}...")
    return list(csv.reader(io.StringIO(text))), sha


def split_header_and_body(rows, a: Audit):
    lens = pd.Series([len(r) for r in rows]).value_counts().to_dict()
    blank = sum(1 for r in rows if not any(c.strip() for c in r))
    ragged = [(i + 1, len(r)) for i, r in enumerate(rows) if any(c.strip() for c in r) and len(r) != 18]
    a.add("S01", "Column count per row", PASS if not ragged else FAIL,
          f"Field counts per row: {lens}. Blank lines: {blank}. Rows with a field count other than 18: {len(ragged)} (first: {ragged[:5]}).")
    rows = [r for r in rows if any(c.strip() for c in r)]
    if len(rows) < 3:
        a.add("S02", "Header rows", FAIL, "Fewer than 3 non-blank rows.")
        return None
    r0, r1 = [c.strip() for c in rows[0]], [c.strip() for c in rows[1]]
    ok0 = r0[:3] == ID_HEADERS
    ok1 = r1[3:18] == SUBHEADINGS and not any(r1[:3])
    a.add("S02", "Two-row header matches the contract", PASS if ok0 and ok1 else FAIL,
          f"row0 id headers {'OK' if ok0 else 'DIFFER: ' + str(r0[:3])}; row1 sub-headings {'OK' if ok1 else 'DIFFER'}"
          + ("" if ok1 else f" (expected {SUBHEADINGS}, found {r1[3:]})"))
    a.add("S03", "Group headers sit over the right blocks", PASS if all(r0[c].lower().startswith(p) for c, p in
          {3: "drive end", 6: "drive end", 9: "non drive end", 12: "non drive end"}.items() if c < len(r0)) else FAIL,
          f"row0 columns 3,6,9,12 = {[r0[c] for c in (3, 6, 9, 12) if c < len(r0)]}. 'Header'/'Vrtex' as typed are ASSUMED to mean horizontal/vertical (unconfirmed).")
    body = pd.DataFrame([r + [""] * (18 - len(r)) if len(r) < 18 else r[:18] for r in rows[2:]],
                        columns=["record_id", "motor_id", "ts_raw"] + SUBHEADINGS).apply(lambda s: s.str.strip())
    body.insert(0, "file_row", range(3, 3 + len(body)))      # 1-based line number among non-blank rows
    a.add("S04", "Data rows", PASS if len(body) == INSPECTED["rows"] else WARN,
          f"{len(body)} data rows (earlier inspection: {INSPECTED['rows']}).")
    return body


# ----------------------------------------------------------------------------------------------
# 2. Identifiers
# ----------------------------------------------------------------------------------------------
def check_ids(b, a: Audit):
    rid, mid = b["record_id"], b["motor_id"]
    empty_r, empty_m = int((rid == "").sum()), int((mid == "").sum())
    dup = rid[rid.duplicated(keep=False) & (rid != "")]
    a.add("I01", "Record IDs present and unique", PASS if not (empty_r or len(dup)) else FAIL,
          f"empty: {empty_r}; duplicated values: {dup.nunique()} covering {len(dup)} rows; examples: {sorted(dup.unique())[:5]}")
    bad_pat = rid[(rid != "") & ~rid.str.fullmatch(r"REC\d{6}")]
    a.add("I02", "Record ID pattern", PASS if bad_pat.empty else WARN, f"{len(bad_pat)} IDs do not match REC######; examples {bad_pat.head(5).tolist()}")
    nums = pd.to_numeric(rid.str.extract(r"(\d+)$")[0], errors="coerce")
    gaps = int((nums.diff().dropna() != 1).sum())
    a.add("I03", "Record ID sequence follows file order", PASS if gaps == 0 else INFO,
          f"{gaps} places where the ID does not increase by 1 in file order. Because the file is ordered motor-then-time, a record number "
          f"encodes motor and time: it must never be a model input.")
    norm = mid.str.upper().str.replace(r"[\s_\-]", "", regex=True)
    variants = {k: sorted(set(mid[norm == k])) for k in norm.unique() if len(set(mid[norm == k])) > 1}
    ws = sorted(set(mid[mid != mid.str.strip()]))
    a.add("M01", "Motor IDs present and consistently spelled", PASS if not (empty_m or variants) else FAIL,
          f"empty: {empty_m}; IDs that collapse together after removing case/space/dash: {variants}")
    per = mid[mid != ""].value_counts()
    fam = mid.str.replace(r"\d+$", "", regex=True)
    a.add("M02", "Motors and records per motor", PASS if per.min() >= 20 else WARN,
          f"{mid.nunique()} motors (earlier inspection: {INSPECTED['motors']}); records per motor min/median/max = {per.min()}/{int(per.median())}/{per.max()}; "
          f"motor families by ID prefix: {fam.groupby(fam).apply(lambda s: mid[s.index].nunique()).to_dict()}")
    return fam


# ----------------------------------------------------------------------------------------------
# 3. Timestamps and chronology
# ----------------------------------------------------------------------------------------------
def _parse(v, fmt):
    try:
        return datetime.strptime(v, fmt)
    except ValueError:
        return None


def check_timestamps(b, a: Audit):
    raw = b["ts_raw"]
    mdy, dmy = raw.map(lambda v: _parse(v, TS_MDY)), raw.map(lambda v: _parse(v, TS_DMY))
    empty = int((raw == "").sum())
    neither = b[(raw != "") & mdy.isna() & dmy.isna()]
    only_m, only_d = int((mdy.notna() & dmy.isna()).sum()), int((dmy.notna() & mdy.isna()).sum())
    both = int((mdy.notna() & dmy.notna()).sum())
    a.add("T01", "Every timestamp parses", PASS if not (empty or len(neither)) else FAIL,
          f"empty: {empty}; unparseable by BOTH month-first and day-first: {len(neither)} (examples {neither.ts_raw.head(5).tolist()}; file rows {neither.file_row.head(5).tolist()})")
    # decisive evidence for month-first vs day-first: chronological order within each motor in FILE order
    def decreases(series):
        n = 0
        for m, idx in b.groupby("motor_id").groups.items():
            t = series.loc[idx].dropna()
            n += int((t.diff().dropna() < pd.Timedelta(0)).sum()) if len(t) > 1 else 0
        return n
    tm, td = pd.to_datetime(mdy, errors="coerce"), pd.to_datetime(dmy, errors="coerce")
    dec_m, dec_d = decreases(tm), decreases(td)
    verdict = ("MONTH-first supported (fewer order violations)" if dec_m < dec_d else "DAY-first supported" if dec_d < dec_m else "not decided by file order")
    a.add("T02", "Day/month order (month-first assumed by the notebook)", PASS if only_d == 0 and dec_m <= dec_d else WARN,
          f"rows readable only as day>12 in 2nd field (month-first only): {only_m}; only as day-first: {only_d}; ambiguous (both valid): {both}. "
          f"File-order decreases within a motor: month-first={dec_m}, day-first={dec_d} -> {verdict}. Still confirm with the data owner.",
          {"decreases_month_first": dec_m, "decreases_day_first": dec_d})
    a.add("T03", "One timestamp convention in the whole file", FAIL if only_d else PASS,
          f"{only_d} rows can only be day-first and {only_m} can only be month-first." + (" The file mixes conventions." if only_d else " No row contradicts month-first."))
    t = tm.copy(); b = b.assign(t=t)
    if t.notna().any():
        now = pd.Timestamp.now()
        a.add("T04", "Timestamp range is plausible", PASS if (t.max() <= now and t.min().year >= 2000) else WARN,
              f"{t.min()} to {t.max()} ({(t.max() - t.min()).days} days). In the future: {int((t > now).sum())}. "
              f"No timezone and minute resolution only (INFO): ties and daylight-saving shifts cannot be excluded.")
    dsame = b[b.duplicated(["motor_id", "t"], keep=False) & b.t.notna()]
    a.add("T05", "No two readings of one motor share a timestamp", PASS if dsame.empty else FAIL,
          f"{len(dsame)} rows share motor+timestamp: file rows {dsame.file_row.head(10).tolist()}")
    cross = int(b.duplicated("t", keep=False).sum()) - len(dsame)
    a.add("T06", "Timestamp collisions across different motors", INFO, f"{cross} rows share a timestamp with another motor (normal if rounds are logged together).")
    out_of_order, ties, gaps_all = [], 0, []
    for m, g in b.groupby("motor_id", sort=False):
        d = g["t"].diff().dropna()
        if (d < pd.Timedelta(0)).any():
            out_of_order.append((m, int((d < pd.Timedelta(0)).sum())))
        ties += int((d == pd.Timedelta(0)).sum()); gaps_all += list(d[d > pd.Timedelta(0)].dt.total_seconds() / 86400)
    a.add("T07", "Chronological order within each motor (file order)", PASS if not out_of_order else FAIL,
          f"motors with a reading earlier than the previous one: {out_of_order[:8]}. Notebook sorts by motor+time+record_id, so sorting hides this from its own checks.")
    g = np.array(gaps_all) if gaps_all else np.array([np.nan])
    a.add("T08", "Sampling regularity", INFO if np.nanmax(g) <= 21 else WARN,
          f"gap between consecutive readings of a motor: median {np.nanmedian(g):.1f} d, 95th pct {np.nanpercentile(g, 95):.1f} d, max {np.nanmax(g):.1f} d; "
          f"gaps > 14 d: {int((g > 14).sum())}. Readings are periodic rounds, not continuous logging.")
    return b


# ----------------------------------------------------------------------------------------------
# 4. Numbers, units, duplicates
# ----------------------------------------------------------------------------------------------
NUM_RE = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$")


def check_numbers(b, a: Audit):
    X = pd.DataFrame(index=b.index)
    problems, empties, sent, implaus, neg = {}, {}, {}, {}, {}
    for raw_name, (clean, unit) in SENSORS.items():
        s = b[raw_name]
        e = int((s == "").sum()); empties[raw_name] = e
        bad = s[(s != "") & ~s.map(lambda v: bool(NUM_RE.match(v)))]
        if len(bad):
            problems[raw_name] = bad.value_counts().head(5).to_dict()
        v = pd.to_numeric(s.where(s.map(lambda x: bool(NUM_RE.match(x)))), errors="coerce")
        X[clean] = v
        lo, hi = PLAUSIBLE[unit]
        sent[raw_name] = int(v.isin(SENTINELS).sum())
        implaus[raw_name] = int(((v < lo) | (v > hi)).sum()) + int((v == 0).sum() if unit in ("mm/s", "gE", "rpm") else 0)
        neg[raw_name] = int((v < 0).sum())
    a.add("N01", "No missing measurement cells", PASS if not any(empties.values()) else FAIL, f"empty cells per column: { {k: v for k, v in empties.items() if v} or 'none' }")
    a.add("N02", "Every non-empty cell is a plain number", PASS if not problems else FAIL, f"non-numeric text: {problems or 'none'} (NaN/inf/'N/A'/'-' strings are caught here)")
    a.add("N03", "No sentinel codes (e.g. -999, 9999) hiding as numbers", PASS if not any(sent.values()) else WARN, f"counts: { {k: v for k, v in sent.items() if v} or 'none' }")
    a.add("N04", "No negative physical readings", PASS if not any(neg.values()) else FAIL, f"negative counts: { {k: v for k, v in neg.items() if v} or 'none' }")
    a.add("N05", "Generic plausibility screen (review only, nothing removed)", PASS if not any(implaus.values()) else WARN,
          f"values outside {PLAUSIBLE} or exactly zero (mm/s, gE, rpm): { {k: v for k, v in implaus.items() if v} or 'none' }")
    desc = X.describe().T[["min", "50%", "max"]].round(3)
    a.tables["numeric_ranges"] = desc.reset_index().rename(columns={"index": "feature"}).to_dict("records")
    # constant / stuck sensors
    const_cols = [c for c in CLEAN if X[c].nunique(dropna=True) <= 1]
    stuck = 0
    for m, g in b.groupby("motor_id"):
        v = X.loc[g.index, CLEAN]
        stuck += int((v.diff().iloc[1:].abs().sum(axis=1) == 0).sum())
    a.add("N06", "No constant column or stuck repeated readings", PASS if not const_cols and stuck == 0 else WARN,
          f"constant columns: {const_cols or 'none'}; consecutive identical full 14-value readings of the same motor: {stuck}")
    return X


def check_units(b, X, fam, a: Audit):
    med = X.groupby(b["motor_id"]).median()
    sensor_cols = [c for c in CLEAN if c not in ("rotational_speed_rpm", "load_percent")]   # operating conditions are covered by U02
    z = (med[sensor_cols] - med[sensor_cols].median()) / (1.4826 * (med[sensor_cols] - med[sensor_cols].median()).abs().median().replace(0, np.nan))
    flagged = {m: {c: round(float(z.loc[m, c]), 1) for c in sensor_cols if abs(z.loc[m, c]) > 6} for m in med.index}
    flagged = {m: v for m, v in flagged.items() if v}
    ratio = (med[[c for c in CLEAN if "velocity" in c]] / med[[c for c in CLEAN if "velocity" in c]].median())
    off = ratio[(ratio < 0.2) | (ratio > 5)].stack().to_dict()
    a.add("U01", "Motors whose typical reading is far from the other motors (possible unit/scale problem)", PASS if not flagged and not off else WARN,
          f"robust z>6 (motor median vs median of motors): {flagged or 'none'}; velocity medians <0.2x or >5x the fleet: {off or 'none'}. "
          f"A WARN is a question for the data owner (different unit? different sensor? genuinely different motor?), not proof of error.")
    fam_tbl = X.groupby(fam).median().round(3)
    a.tables["family_medians"] = fam_tbl.reset_index().rename(columns={"motor_id": "family", 0: "family"}).to_dict("records")
    rpm = b.assign(rpm=X["rotational_speed_rpm"]).groupby("motor_id")["rpm"].median()
    a.add("U02", "Rotational speed groups", INFO,
          f"per-motor median rpm min/max {rpm.min():.0f}/{rpm.max():.0f}; histogram by 250-rpm bins {np.histogram(rpm, bins=range(0, 3500, 250))[0].tolist()}. "
          f"Clusters near 1500 and 3000 would indicate 4-pole and 2-pole machines (different physics). Not stated in the file.")


def check_duplicates(b, X, a: Audit):
    cols = ["motor_id", "ts_raw"] + CLEAN
    full = int(pd.concat([b[["motor_id", "ts_raw"]], X], axis=1).duplicated().sum())
    same_vec = int(X.duplicated(keep=False).sum())
    a.add("D01", "Duplicate records", PASS if full == 0 else FAIL, f"fully identical (ignoring record id): {full}; rows with an identical 14-value sensor vector: {same_vec}")
    conflict = b.assign(**{c: X[c] for c in CLEAN})
    conflict = conflict[conflict.duplicated(CLEAN, keep=False)].groupby(CLEAN)[LABEL_RAW].nunique()
    a.add("D02", "Identical readings with different labels", PASS if (conflict > 1).sum() == 0 else WARN, f"{int((conflict > 1).sum())} identical sensor vectors carry more than one label (label noise)")


# ----------------------------------------------------------------------------------------------
# 5. Labels and targets
# ----------------------------------------------------------------------------------------------
def check_labels(b, X, fam, a: Audit):
    lab = b[LABEL_RAW]
    unknown = sorted(set(lab[lab != ""]) - set(CLASSES)); empty = int((lab == "").sum())
    a.add("L01", "Label values are exactly Normal / Alert / Danger", PASS if not unknown and not empty else FAIL, f"unknown values: {unknown or 'none'}; empty: {empty}")
    cnt = lab.value_counts().reindex(CLASSES).fillna(0).astype(int)
    exp = INSPECTED["labels"]
    a.add("L02", "Class distribution", PASS if cnt.to_dict() == exp else WARN,
          f"counts {cnt.to_dict()} ({(cnt / cnt.sum() * 100).round(1).to_dict()} %); earlier inspection {exp}; imbalance largest/smallest = {cnt.max() / max(cnt.min(), 1):.1f}:1")
    sev = lab.map({c: i for i, c in enumerate(CLASSES)})
    tab = pd.crosstab(b["motor_id"], lab).reindex(columns=CLASSES, fill_value=0)
    top3 = tab["Danger"].sort_values(ascending=False).head(3)
    a.add("L03", "How concentrated are Danger labels?", INFO if top3.sum() < 0.6 * max(tab["Danger"].sum(), 1) else WARN,
          f"motors with any Danger: {int((tab.Danger > 0).sum())} of {len(tab)}; top-3 motors hold {int(top3.sum())} of {int(tab.Danger.sum())} Danger records ({top3.to_dict()}). "
          f"Few motors carrying most Danger records makes any motor-grouped score highly dependent on which motors land in test.")
    famtab = pd.crosstab(fam, lab).reindex(columns=CLASSES, fill_value=0)
    famtab["Danger_%"] = (famtab["Danger"] / famtab.sum(axis=1) * 100).round(1)
    a.tables["labels_by_family"] = famtab.reset_index().rename(columns={"motor_id": "family"}).to_dict("records")
    a.add("L04", "Label mix differs by motor family", WARN if famtab["Danger_%"].max() - famtab["Danger_%"].min() > 5 else INFO,
          f"Danger % by family: {famtab['Danger_%'].to_dict()}. If families differ this much, 'family' is a hidden predictor and a random motor split does not test generalisation to a new family.")
    flips, impr, const = 0, 0, 0
    for m, g in b.assign(sev=sev, t=b.get("t")).groupby("motor_id"):
        s = g["sev"].to_numpy()
        flips += int((np.diff(s) != 0).sum()); impr += int((np.diff(s) < 0).sum()); const += int(len(set(s)) == 1)
    a.add("L05", "Label dynamics inside a motor (file order)", INFO,
          f"label changes: {flips}; improvements (e.g. Danger->Normal): {impr}; motors with one constant label: {const}. Improvements may mean repair, re-labelling or noise: ask the data owner.")


def check_targets(rows, b, X, a: Audit):
    header_cells = [c.strip() for r in rows[:2] for c in r if c.strip()]
    kw = re.compile(r"fail|event|repair|maint|breakdown|rul|horizon|next|within|7.?day|seven|remain", re.I)
    hits = [c for c in header_cells if kw.search(c)]
    a.tables["column_inventory"] = [{"column": c} for c in header_cells]
    a.add("G01", "Is a failure-event or 7-day failure target present in the file?", FAIL if not hits else INFO,
          (f"No header contains failure/event/repair/maintenance/horizon words. The ONLY target-like column is '{LABEL_RAW}' (current condition). "
           f"A '7-day failure' target does NOT exist in this file and cannot be created from it without confirmed failure events." if not hits else f"Candidate columns: {hits}. Inspect their definition and source."))
    if "t" in b:
        sev = b[LABEL_RAW].map({c: i for i, c in enumerate(CLASSES)})
        pos = cen = tot = 0
        for m, g in b.assign(sev=sev).sort_values(["motor_id", "t"]).groupby("motor_id"):
            t, s = g["t"].to_numpy(), g["sev"].to_numpy()
            for i in range(len(g)):
                tot += 1
                later = (t > t[i]) & (t <= t[i] + np.timedelta64(7, "D"))
                if not later.any():
                    cen += 1
                elif (s[later] == 2).any():
                    pos += 1
        a.add("G02", "PROXY only: 'a later Danger label within 7 days' (NOT a failure label)", INFO,
              f"{pos} of {tot} readings have a later Danger label within 7 days; {cen} have no later reading in the window (censored). "
              f"This counts future LABELS, not failures. Do not call it failure prediction unless Danger is shown to precede real failures.")


# ----------------------------------------------------------------------------------------------
# 6. Leakage and evaluation design
# ----------------------------------------------------------------------------------------------
def propose_grouped_split(df, seed=SEED):
    """Same procedure as notebook v1.0.0 section 11 (uses label counts only)."""
    motors = np.array(sorted(df["motor_id"].unique()))
    M = pd.crosstab(df["motor_id"], df["y"]).reindex(index=motors, columns=[0, 1, 2], fill_value=0).to_numpy()
    total = M.sum(0); glob = total / total.sum(); n = len(motors)
    n_tr, n_va = int(round(SPLIT_FRACTIONS[0] * n)), int(round(SPLIT_FRACTIONS[1] * n))
    rng = np.random.default_rng(seed); best = None; feasible = 0
    for _ in range(N_SPLIT_TRIALS):
        perm = rng.permutation(n); parts = [perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]]
        score, ok = 0.0, True
        for k, idx in enumerate(parts):
            c = M[idx].sum(0)
            if (c < MIN_CLASS_RECORDS).any():
                ok = False; break
            score += np.abs(c / c.sum() - glob).sum() + abs(c.sum() / total.sum() - SPLIT_FRACTIONS[k])
        if ok:
            feasible += 1
            if best is None or score < best[0]:
                best = (score, parts)
    if best is None:
        return None, 0
    return {nm: list(motors[idx]) for nm, idx in zip(["train", "val", "test"], best[1])}, feasible


def make_model_frame(b, X, fam):
    df = X.copy(); df.insert(0, "motor_id", b["motor_id"].values); df["family"] = fam.values
    df["y"] = b[LABEL_RAW].map({c: i for i, c in enumerate(CLASSES)}).values
    if "t" in b:
        df["t"] = b["t"].values
    return df


def check_leakage_and_split(b, X, fam, a: Audit, fast=False):
    try:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import f1_score
        from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
        from sklearn.tree import DecisionTreeClassifier
    except ImportError:
        a.add("K00", "scikit-learn available", SKIP, "Install scikit-learn to run leakage screens.")
        return None
    if X.isna().any().any() or b[LABEL_RAW].isin(CLASSES).sum() != len(b):
        a.add("K00", "Leakage screens", SKIP, "Skipped because missing/invalid inputs or labels exist (fix those first; nothing was imputed).")
        return None
    df = make_model_frame(b, X, fam)
    y, g = df["y"].to_numpy(), df["motor_id"].to_numpy()
    # K01: file order / record number as a predictor (file is ordered motor then time)
    idx = np.arange(len(df)).reshape(-1, 1)
    p = cross_val_predict(DecisionTreeClassifier(max_depth=6, random_state=0), idx, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    maj = max(np.bincount(y)) / len(y)
    a.add("K01", "Record number alone predicts the label (must stay out of features)", WARN if (p == y).mean() > maj + 0.03 else PASS,
          f"accuracy using ONLY file position: {(p == y).mean():.3f} vs majority baseline {maj:.3f}. A gain means row order carries motor/time information.")
    # K02: per-motor majority label = ceiling for memorising motor identity
    mm = df.groupby("motor_id")["y"].agg(lambda s: s.value_counts().idxmax())
    a.add("K02", "Motor identity alone as a predictor (memorisation ceiling)", INFO,
          f"always predicting each motor's own majority label scores {(df['motor_id'].map(mm) == df['y']).mean():.3f} (majority-class baseline {maj:.3f}). "
          f"This is why 'motor_id' is excluded and the split must keep a motor's rows together.")
    # K03: can sensors identify the motor / family? (fingerprint)
    Xv = df[FEATURES].to_numpy()
    rf = RandomForestClassifier(n_estimators=20 if fast else 150, random_state=0, n_jobs=-1)
    pm = cross_val_predict(rf, Xv, g, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    pf = cross_val_predict(rf, Xv, df["family"].to_numpy(), cv=StratifiedKFold(5, shuffle=True, random_state=0))
    a.add("K03", "Sensors fingerprint the motor / family", WARN if (pm == g).mean() > 0.5 else INFO,
          f"motor identification accuracy from the 14 inputs: {(pm == g).mean():.3f} (chance {1 / df.motor_id.nunique():.3f}); family: {(pf == df['family']).mean():.3f} (chance {1 / df.family.nunique():.3f}). "
          f"High values mean readings from one motor are near-duplicates, so a random ROW split would leak.")
    # K04: label-rule probe on the WHOLE file with motor-grouped CV (diagnostic; no model selection involved)
    out = {}
    for name, cols in {"velocity only": [c for c in FEATURES if "velocity" in c], "gE only": [c for c in FEATURES if c.endswith("_ge")],
                       "temperature only": [c for c in FEATURES if "temp" in c], "operating only": ["rotational_speed_rpm", "load_percent"]}.items():
        best = 0
        for depth in (1, 2, 3):
            pr = cross_val_predict(DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=0), df[cols].to_numpy(), y, cv=GroupKFold(5), groups=g)
            best = max(best, f1_score(y, pr, average="macro"))
        out[name] = round(best, 3)
    a.add("K04", "Can a tiny rule on ONE sensor type reproduce the labels?", WARN if out["velocity only"] >= 0.9 or max(out.values()) >= 0.9 else INFO,
          f"best motor-grouped macro-F1 of a depth<=3 tree: {out}. Near 1.0 for one sensor type suggests the labels were DERIVED from that sensor (circularity). "
          f"Moderate values do not prove independence. Provenance from the data owner is still required.")
    # K05: operating columns as motor-type proxy
    within = df.groupby("motor_id")["rotational_speed_rpm"].std().median(); between = df.groupby("motor_id")["rotational_speed_rpm"].median().std()
    a.add("K05", "Operating conditions act like a motor-type ID", WARN if between > 5 * max(within, 1e-9) else INFO,
          f"rpm spread within a motor (median SD) {within:.1f} vs between motor medians {between:.1f}")
    # SPLIT
    split, feas = propose_grouped_split(df)
    if split is None:
        a.add("P01", "Reproduce notebook motor-grouped split", FAIL, "No partition satisfies the minimum-class rule with these labels.")
        return df, None
    comp = {}
    for s, ms in split.items():
        d = df[df.motor_id.isin(ms)]
        comp[s] = {"motors": len(ms), "records": len(d), **{c: int((d.y == i).sum()) for i, c in enumerate(CLASSES)},
                   "families": d.family.value_counts().to_dict(), "danger_in_top_motor_%": round(100 * d[d.y == 2].motor_id.value_counts().iloc[0] / max((d.y == 2).sum(), 1), 1)}
    a.tables["reproduced_split"] = comp
    a.add("P01", "Reproduced motor-grouped split (notebook v1.0.0 procedure, seed 42)", INFO,
          f"{feas} of {N_SPLIT_TRIALS} candidate partitions feasible. Compare this table with the one the notebook printed; they must match. {json.dumps(comp)}")
    fams_test = set(df[df.motor_id.isin(split['test'])].family)
    a.add("P02", "Split vs. research claim", WARN,
          f"A motor-grouped random split supports ONLY: 'does the model classify the recorded condition of UNSEEN MOTORS drawn from the SAME five families in the SAME Apr-Aug window'. "
          f"It does NOT support: (a) predicting FUTURE failures (no future-event target), (b) a NEW motor family or plant (test families {sorted(fams_test)} also appear in training), "
          f"(c) performance after time passes (train and test cover the same dates).")
    # leave-one-family-out table to quantify the family effect
    rows = []
    for f_ in ([] if fast else sorted(df.family.unique())):
        tr, te = df[df.family != f_], df[df.family == f_]
        r = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1, class_weight="balanced").fit(tr[FEATURES], tr.y)
        pr = r.predict(te[FEATURES])
        rows.append({"held_out_family": f_, "n": len(te), "danger_in_family": int((te.y == 2).sum()),
                     "macro_F1": round(float(f1_score(te.y, pr, average="macro", labels=[0, 1, 2], zero_division=0)), 3)})
    a.tables["leave_one_family_out_RF"] = rows
    a.add("P03", "Leave-one-FAMILY-out sensitivity (Random Forest, diagnostic)", SKIP if fast else INFO,
          "skipped in fast mode" if fast else
          f"{rows}. Large drops versus the motor-grouped score mean the headline number depends on the family being in training.")
    return df, split


# ----------------------------------------------------------------------------------------------
# 7. Bundle (training <-> inference contract)
# ----------------------------------------------------------------------------------------------
def numpy_forward(layers, mean, scale, X):
    z = (np.asarray(X, dtype="float64") - mean) / scale
    for W, bb, act in layers:
        z = z @ W + bb
        z = np.maximum(z, 0) if act == "relu" else (np.exp(z - z.max(1, keepdims=True)) / np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    return z


def check_bundle(bundle_dir, raw_sha, df, split, a: Audit):
    d = Path(bundle_dir)
    need = ["preprocessing.json", "feature_schema.json", "class_mapping.json", "model_architecture.json", "weights.npz", "training_metadata.json", "parity_probe.json", "MANIFEST.json"]
    miss = [n for n in need if not (d / n).exists()]
    if miss:
        a.add("B01", "Bundle files present", FAIL, f"missing: {miss}")
        return
    J = lambda n: json.loads((d / n).read_text(encoding="utf-8"))
    man = J("MANIFEST.json")
    bad = [n for n, h in man["files"].items() if not (d / n).exists() or sha256_of(d / n) != h]
    a.add("B01", "Bundle files match MANIFEST checksums", PASS if not bad else FAIL, f"changed or missing: {bad or 'none'}")
    pre, sch, cls, arch, meta, par = J("preprocessing.json"), J("feature_schema.json"), J("class_mapping.json"), J("model_architecture.json"), J("training_metadata.json"), J("parity_probe.json")
    names = pre["feature_names"]
    a.add("B02", "Feature names and ORDER: training == bundle == schema", PASS if names == FEATURES and [f["name"] for f in sch["features"]] == FEATURES else FAIL,
          f"preprocessing={names == FEATURES}; schema={[f['name'] for f in sch['features']] == FEATURES}")
    units_ok = all(f["unit"] == SENSORS[r][1] for f, r in zip(sch["features"], SENSORS) if len(sch["features"]) == len(SENSORS))
    a.add("B03", "Units in feature_schema match the file's column units", PASS if units_ok else FAIL, "units compared column by column")
    a.add("B04", "Class order", PASS if [c["name"] for c in sorted(cls["classes"], key=lambda c: c["id"])] == CLASSES else FAIL, f"bundle classes {cls['classes']}")
    ds = (meta.get("dataset") or {})
    a.add("B05", "Model was trained on THIS data file", PASS if ds.get("sha256") == raw_sha else FAIL,
          f"metadata sha256 {str(ds.get('sha256'))[:16]}... vs file {raw_sha[:16]}...")
    a.add("B06", "Not a smoke-test bundle", PASS if not meta.get("fast_debug_run") else FAIL, f"fast_debug_run={meta.get('fast_debug_run')}")
    lp = meta.get("label_provenance") or {}
    todo = [k for k, v in lp.items() if k != "documented" and (v is None or (isinstance(v, str) and v.strip().upper().startswith("TODO")))]
    prov_flag = meta.get("provisional_label_provenance")
    if prov_flag is False and todo:
        a.add("B07", "Provenance flag is justified by evidence", FAIL, f"bundle says provenance is DOCUMENTED (provisional=False) but these fields are still empty/TODO: {todo}. The honesty gate was switched off without evidence.")
    elif prov_flag is False:
        a.add("B07", "Provenance flag is justified by evidence", WARN, f"Marked documented: {lp}. A program cannot verify this; check the evidence_reference document yourself.")
    else:
        a.add("B07", "Provenance flag is justified by evidence", PASS, "Bundle is stamped provisional (provenance not documented). This is the honest state if no written evidence exists.")
    # scaler fitted on TRAIN only? needs the reproduced split
    mean, scale = np.array(pre["mean"]), np.array(pre["scale"])
    if df is not None and split is not None:
        tr = df[df.motor_id.isin(split["train"])][FEATURES].to_numpy(); al = df[FEATURES].to_numpy()
        e_tr = float(np.abs(mean - tr.mean(0)).max() / (np.abs(tr.mean(0)).max() + 1e-12)); e_al = float(np.abs(mean - al.mean(0)).max() / (np.abs(al.mean(0)).max() + 1e-12))
        s_tr = float(np.abs(scale - tr.std(0, ddof=0)).max()); s_al = float(np.abs(scale - al.std(0, ddof=0)).max())
        ok_tr = e_tr < 1e-6 and s_tr < 1e-6
        ok_al = e_al < 1e-6 and s_al < 1e-6
        a.add("B08", "Scaler parameters were learned from TRAINING motors only", PASS if ok_tr and not ok_al else FAIL if ok_al else WARN,
              f"max relative mean diff vs reproduced-train {e_tr:.2e} / vs all-data {e_al:.2e}; max scale diff {s_tr:.2e} / {s_al:.2e}. "
              + ("Matches training motors only." if ok_tr and not ok_al else "MATCHES ALL DATA: test/validation information leaked into scaling." if ok_al else
                 "Matches neither: the reproduced split or seed differs from the one actually used (compare split tables)."))
        sm = (meta.get("split") or {}).get("summary") or {}
        rep = {s: {"motors": len(split[s]), "records": int(df.motor_id.isin(split[s]).sum())} for s in split}
        a.add("B09", "Split recorded in the bundle equals the reproduced split", PASS if sm == rep else WARN, f"bundle {sm} vs reproduced {rep}")
    # inference parity on the artificial probes with an independent forward pass
    z = np.load(d / "weights.npz", allow_pickle=False)
    layers = [(z[f"W{i}"], z[f"b{i}"], s["activation"]) for i, s in enumerate(arch["layers"])]
    worst = 0.0
    for p in par["probes"]:
        got = numpy_forward(layers, mean, scale, np.array([[p["readings"][n] for n in names]]))[0]
        worst = max(worst, float(np.abs(got - np.array([p["expected_probabilities"][c] for c in CLASSES])).max()))
    a.add("B10", "Independent NumPy forward pass reproduces the notebook's probe outputs", PASS if worst < 1e-5 else FAIL, f"max abs difference {worst:.2e} over {len(par['probes'])} probes (artificial inputs: not real rows)")


# ----------------------------------------------------------------------------------------------
# 8. Cleaned copy equals the original (nothing altered)
# ----------------------------------------------------------------------------------------------
def check_clean_copy(path, b, X, a: Audit):
    c = pd.read_csv(path, dtype=str, keep_default_na=False)
    exp = ["record_id", "motor_id", "measured_at"] + CLEAN + ["condition_label"]
    a.add("C01", "Clean copy has the expected columns", PASS if list(c.columns) == exp else FAIL, f"columns {list(c.columns)}")
    a.add("C02", "No row dropped or added", PASS if len(c) == len(b) and set(c.record_id) == set(b.record_id) else FAIL, f"clean {len(c)} vs raw {len(b)}")
    m = c.set_index("record_id").join(b.set_index("record_id")[["motor_id", LABEL_RAW]].rename(columns={"motor_id": "mid_raw", LABEL_RAW: "lab_raw"}), how="inner")
    Xi = X.set_axis(b["record_id"].values)
    num_diff = int((~np.isclose(m[CLEAN].astype(float).to_numpy(), Xi.loc[m.index, CLEAN].to_numpy(), rtol=0, atol=0)).sum()) if len(m) else -1
    a.add("C03", "Every sensor value is identical to the original (no cleaning, rounding or imputation)", PASS if num_diff == 0 else FAIL, f"differing cells: {num_diff}")
    a.add("C04", "Motor IDs and labels identical to the original", PASS if (m.motor_id == m.mid_raw).all() and (m.condition_label == m.lab_raw).all() else FAIL, "compared row by row")


# ----------------------------------------------------------------------------------------------
def guarded(a: Audit, stage, fn, *args, **kw):
    """Run one stage; if it crashes on malformed input, record a FAIL instead of stopping the whole audit."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001 - deliberate: a crash on bad input is itself a finding
        a.add(stage, f"Stage '{fn.__name__}' could not complete", FAIL, f"{type(exc).__name__}: {exc}. The input is malformed in a way this stage cannot read; fix it and re-run.")
        return None


def run(data, out, clean_copy=None, bundle=None, fast=False):
    a = Audit()
    sha_before = sha256_of(data)
    rows, sha = read_text_grid(data, a)
    b = split_header_and_body(rows, a)
    if b is None:
        return a, sha_before
    fam = check_ids(b, a)
    fam = fam.set_axis(b.index)
    b = check_timestamps(b, a)
    X = check_numbers(b, a)
    guarded(a, "U99", check_units, b, X, fam, a)
    guarded(a, "D99", check_duplicates, b, X, a)
    guarded(a, "L99", check_labels, b, X, fam, a)
    guarded(a, "G99", check_targets, rows, b, X, a)
    df, split = (guarded(a, "K99", check_leakage_and_split, b, X, fam, a, fast) or (None, None))
    if clean_copy:
        guarded(a, "C99", check_clean_copy, clean_copy, b, X, a)
    if bundle:
        guarded(a, "B99", check_bundle, bundle, sha, df, split, a)
    a.add("Z01", "Input file unchanged by this audit", PASS if sha256_of(data) == sha_before else FAIL, f"SHA-256 before = after = {sha_before[:16]}...")
    a.add("Z02", "Label provenance", WARN,
          "UNVERIFIED by any program. Who/what assigned Normal/Alert/Danger, from which data, and whether it was confirmed by maintenance records "
          "must come from written evidence supplied by the data owner. Until then treat every label as unverified.")
    return a, sha_before


def verdict(a: Audit):
    fails = [f for f in a.findings if f["status"] == FAIL]
    if fails:
        return "NOT VALIDATED: " + str(len(fails)) + " check(s) FAILED (" + ", ".join(f["id"] for f in fails) + ")"
    return "STRUCTURE CHECKS PASSED, but LABEL PROVENANCE IS UNVERIFIED. This is not a declaration that the dataset is valid."


def write_reports(a: Audit, out: Path, data):
    out.mkdir(parents=True, exist_ok=True)
    meta = {"audit_version": AUDIT_VERSION, "run_at": datetime.now().isoformat(timespec="seconds"), "input": str(data), "verdict": verdict(a)}
    (out / "audit_report.json").write_text(json.dumps({**meta, "findings": a.findings, "tables": a.tables}, indent=2, default=str), encoding="utf-8")
    L = [f"# Dataset audit report", f"- Run: {meta['run_at']} | audit v{AUDIT_VERSION} | input: `{Path(data).name}`", f"- **Verdict: {meta['verdict']}**", "",
         "| ID | Status | Check | Detail |", "|---|---|---|---|"]
    for f in a.findings:
        L.append(f"| {f['id']} | **{f['status']}** | {f['title']} | {str(f['detail']).replace('|', '/')} |")
    for k, v in a.tables.items():
        L += ["", f"### {k}", "```", json.dumps(v, indent=1, default=str)[:4000], "```"]
    (out / "audit_report.md").write_text("\n".join(L), encoding="utf-8")
    return meta


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True, help="raw master CSV (read-only)")
    ap.add_argument("--clean-copy", help="motor_clean_working_copy.csv written by the notebook")
    ap.add_argument("--bundle", help="folder of the exported model bundle")
    ap.add_argument("--fast", action="store_true", help="smaller models for the leakage screens (smoke tests only)")
    ap.add_argument("--out", default="audit_outputs", help="where reports are written (never next to the original unless you choose so)")
    args = ap.parse_args(argv)
    if not Path(args.data).exists():
        print("File not found:", args.data); return 2
    if Path(args.out).resolve() == Path(args.data).resolve().parent:
        print("Refusing to write reports into the folder that holds the original. Choose another --out."); return 2
    a, _ = run(args.data, Path(args.out), args.clean_copy, args.bundle, args.fast)
    meta = write_reports(a, Path(args.out), args.data)
    sym = {FAIL: "FAIL", WARN: "WARN", PASS: "pass", INFO: "info", SKIP: "skip"}
    for f in a.findings:
        print(f"[{sym[f['status']]:4}] {f['id']}  {f['title']}")
    print("\n" + meta["verdict"]); print("Reports:", Path(args.out).resolve())
    return 1 if any(f["status"] == FAIL for f in a.findings) else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile pm_model_validation.py
#!/usr/bin/env python3
"""pm_model_validation.py - independent re-evaluation of the EXPORTED model bundle (read-only).

It does NOT train a replacement model. It loads what the training notebook saved, rebuilds the held-out test
motors from the raw data with the notebook's deterministic split, and checks whether the saved artefacts
reproduce the numbers the notebook recorded. Baselines are re-fitted ONLY with the parameters the notebook
recorded, on the same training motors, to compare under the same protocol.

Every number it prints is computed from YOUR files at run time. If a required file is missing, the check says
NOT VALIDATED and explains what is missing. Nothing is estimated or filled in.

USAGE
  python pm_model_validation.py --data RAW.csv --bundle pm_condition_ann_v1.0.0 --out validation_outputs
  optional: --events verified_failure_events.csv   (Target B feasibility only; no Target B model exists)
            --backend-dir path/to/backend           (compare the backend's inference code with the bundle)
Exit code: 0 = no FAIL, 1 = at least one FAIL, 2 = could not run.
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import re
import subprocess
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import pm_data_audit as A

VALIDATION_VERSION = "1.0.0"
CLASSES = A.CLASSES
FEATURES = A.FEATURES
PASS, FAIL, WARN, INFO, NV = "PASS", "FAIL", "WARN", "INFO", "NOT VALIDATED"
FORBIDDEN = re.compile(r"record|motor_id|measured|timestamp|label|fail|event|repair|maint|split|^y$|time", re.I)


class Report:
    def __init__(self):
        self.checks, self.tables, self.metrics, self.figures = [], {}, {}, []

    def add(self, cid, title, status, detail):
        self.checks.append({"id": cid, "title": title, "status": status, "detail": detail})
        return status

    def status(self, cid):
        return next((c["status"] for c in self.checks if c["id"] == cid), None)


def sha256_of(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


# ------------------------------------------------------------------------------------------ data + bundle
def load_dataset(path):
    a = A.Audit()
    rows, _ = A.read_text_grid(path, a)
    b = A.split_header_and_body(rows, a)
    if b is None or any(f["status"] == "FAIL" and f["id"] in ("S01", "S02", "S03") for f in a.findings):
        raise ValueError("Raw file structure does not match the contract; run pm_data_audit.py first.")
    fam = A.check_ids(b, a).set_axis(b.index)
    b = A.check_timestamps(b, a)
    X = A.check_numbers(b, a)
    if X.isna().any().any() or not b[A.LABEL_RAW].isin(CLASSES).all():
        raise ValueError("Missing/invalid inputs or labels: run pm_data_audit.py and fix those first (nothing is imputed here).")
    df = A.make_model_frame(b, X, fam)
    df.insert(0, "record_id", b["record_id"].values)
    return df.rename(columns={"t": "measured_at"}), sha256_of(path)


def load_bundle(d):
    d = Path(d)
    J = lambda n: json.loads((d / n).read_text(encoding="utf-8"))
    pre, arch, cls, meta, sch = J("preprocessing.json"), J("model_architecture.json"), J("class_mapping.json"), J("training_metadata.json"), J("feature_schema.json")
    with np.load(d / "weights.npz", allow_pickle=False) as z:
        layers = [(z[f"W{i}"].astype("float64"), z[f"b{i}"].astype("float64"), s["activation"]) for i, s in enumerate(arch["layers"])]
    return {"dir": d, "names": pre["feature_names"], "mean": np.array(pre["mean"]), "scale": np.array(pre["scale"]), "layers": layers, "pre": pre,
            "classes": [c["name"] for c in sorted(cls["classes"], key=lambda c: c["id"])], "meta": meta, "schema": sch}


def forward(bun, X):
    z = (np.asarray(X, dtype="float64") - bun["mean"]) / bun["scale"]
    for W, b, act in bun["layers"]:
        z = z @ W + b
        if act == "relu":
            z = np.maximum(z, 0)
        else:
            e = np.exp(z - z.max(1, keepdims=True)); z = e / e.sum(1, keepdims=True)
    return z


# ------------------------------------------------------------------------------------------ metrics
def compute_metrics(y, proba):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, log_loss,
                                 matthews_corrcoef, precision_recall_fscore_support, roc_auc_score)
    y, proba = np.asarray(y), np.asarray(proba); pred = proba.argmax(1); L = [0, 1, 2]
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    cm = confusion_matrix(y, pred, labels=L)
    m = {"n": int(len(y)), "accuracy": float(accuracy_score(y, pred)), "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
         "macro_f1": float(f1_score(y, pred, labels=L, average="macro", zero_division=0)), "weighted_f1": float(f1_score(y, pred, labels=L, average="weighted", zero_division=0)),
         "macro_precision": float(p.mean()), "macro_recall": float(r.mean()), "mcc": float(matthews_corrcoef(y, pred)),
         "weighted_kappa_quadratic": float(cohen_kappa_score(y, pred, weights="quadratic", labels=L)),
         "log_loss": float(log_loss(y, np.clip(proba, 1e-12, 1), labels=L)),
         "brier": float(np.mean(((proba - np.eye(3)[y]) ** 2).sum(1))),
         "per_class": {CLASSES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i in L},
         "confusion_matrix": cm.tolist(), "danger_predicted_as_normal": int(((y == 2) & (pred == 0)).sum())}
    if len(set(y)) == 3:
        m["roc_auc_ovr_macro"] = float(roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=L))
    ap = {CLASSES[k]: float(average_precision_score((y == k).astype(int), proba[:, k])) for k in L if (y == k).any()}
    m["average_precision"] = ap; m["pr_auc_macro"] = float(np.mean(list(ap.values())))
    m["prevalence"] = {CLASSES[k]: float((y == k).mean()) for k in L}
    m["abnormal_recall"] = float(((y > 0) & (pred > 0)).sum() / max((y > 0).sum(), 1)); m["abnormal_precision"] = float(((y > 0) & (pred > 0)).sum() / max((pred > 0).sum(), 1))
    return m


def manual_selfcheck(y, proba, m):
    """Recompute accuracy and per-class precision/recall by hand (NumPy only) and compare with sklearn."""
    pred = np.asarray(proba).argmax(1); y = np.asarray(y); worst = abs(float((pred == y).mean()) - m["accuracy"])
    for k, c in enumerate(CLASSES):
        tp = ((pred == k) & (y == k)).sum(); pr = tp / max((pred == k).sum(), 1); rc = tp / max((y == k).sum(), 1)
        worst = max(worst, abs(pr - m["per_class"][c]["precision"]), abs(rc - m["per_class"][c]["recall"]))
    return float(worst)


def cluster_ci(y, pred, groups, fn, B=2000, seed=0):
    rng = np.random.default_rng(seed); ms = np.unique(groups); idx = {g: np.where(groups == g)[0] for g in ms}; v = []
    for _ in range(B):
        i = np.concatenate([idx[g] for g in rng.choice(ms, len(ms), replace=True)]); v.append(fn(y[i], pred[i]))
    return [float(x) for x in np.percentile(v, [2.5, 97.5])]


# ------------------------------------------------------------------------------------------ the checks
def run(data, bundle_dir, out, events=None, backend_dir=None, seed=42, B=2000, support=None):
    R = Report(); out = Path(out); out.mkdir(parents=True, exist_ok=True)
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score, recall_score
    sha_in = sha256_of(data)
    df, sha = load_dataset(data); bun = load_bundle(bundle_dir); meta = bun["meta"]; bd = Path(bundle_dir)

    # V01 target and features
    ok = bun["names"] == FEATURES and bun["classes"] == CLASSES and bun["layers"][0][0].shape[0] == 14
    R.add("V01", "Exact target and feature columns", PASS if ok else FAIL,
          f"target = 'condition_label' mapped {dict(enumerate(bun['classes']))}; {len(bun['names'])} inputs in this order: {bun['names']}; first-layer input width {bun['layers'][0][0].shape[0]}; matches the 14-input contract: {ok}")
    bad = [n for n in bun["names"] if FORBIDDEN.search(n)]
    R.add("V02", "No ID, target, event or metadata column is a model input", PASS if not bad and ok else FAIL, f"inputs matching identifier/label/time/event/maintenance patterns: {bad or 'none'}")

    # V04 split (needed before anything uses the test set)
    split, feas = A.propose_grouped_split(df, seed)
    rec_ok = True; why = []
    if split is None:
        R.add("V04", "Held-out test set reproducible", FAIL, "No feasible split; cannot define the held-out test motors. Evaluation NOT performed.")
        return finish(R, out, df, None, bun, sha, sha_in, data)
    summ = (meta.get("split") or {}).get("summary")
    rep = {s: {"motors": len(split[s]), "records": int(df.motor_id.isin(split[s]).sum())} for s in split}
    if summ and summ != rep:
        rec_ok = False; why.append(f"metadata summary {summ} != reproduced {rep}")
    rr = bd / "reproducibility_report.md"
    if rr.exists():
        m_ = re.search(r"## Motors per split[^\n]*\n(\{.*?\n\})", rr.read_text(encoding="utf-8"), re.S)
        if m_:
            try:
                recorded = json.loads(m_.group(1))
                if {k: sorted(v) for k, v in recorded.items()} != {k: sorted(v) for k, v in split.items()}:
                    rec_ok = False; why.append("motor lists in reproducibility_report.md differ from the reproduced split")
            except json.JSONDecodeError:
                why.append("could not parse motor lists in reproducibility_report.md")
        else:
            why.append("reproducibility_report.md has no motor lists")
    else:
        why.append("reproducibility_report.md not in the bundle: split could only be compared by counts")
    sp = Path(support) / "split_motors.json" if support else None
    if sp is not None and sp.exists():
        rec_split = json.loads(sp.read_text(encoding="utf-8"))
        if {k: sorted(v) for k, v in rec_split.items()} != {k: sorted(v) for k, v in split.items()}:
            rec_ok = False; why.append("split_motors.json saved by the notebook differs from the reproduced split")
        else:
            why = [w_ for w_ in why if "reproducibility_report.md not in the bundle" not in w_ and "no motor lists" not in w_]
    s_sets = {k: set(v) for k, v in split.items()}
    disj = s_sets["train"].isdisjoint(s_sets["val"]) and s_sets["train"].isdisjoint(s_sets["test"]) and s_sets["val"].isdisjoint(s_sets["test"])
    df["split"] = df.motor_id.map({m: s for s, ms in split.items() for m in ms})
    comp = {s: {"motors": len(split[s]), "records": int((df.split == s).sum()), **{c: int(((df.split == s) & (df.y == i)).sum()) for i, c in enumerate(CLASSES)},
                "families": df[df.split == s].family.value_counts().to_dict()} for s in split}
    R.tables["split"] = comp
    status = FAIL if not (rec_ok and disj) else (WARN if why else PASS)
    R.add("V04", "Split is by motor, reproducible, and matches what the notebook recorded", status,
          f"disjoint motors: {disj}; reproduced = recorded: {rec_ok}. Notes: {why or 'none'}. {json.dumps(comp)}. "
          f"Claim supported: unseen MOTORS of the same five families in the same Apr-Aug 2025 window. NOT supported: future failures, a new family or plant.")
    if status == FAIL:
        R.add("V06", "Held-out evaluation", NV, "Skipped: the test motors cannot be defined unambiguously, and the test set must not be changed to make results come out.")
        return finish(R, out, df, None, bun, sha, sha_in, data)

    tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
    Xtr, ytr, Xte, yte, gte = tr[FEATURES].to_numpy(float), tr.y.to_numpy(), te[FEATURES].to_numpy(float), te.y.to_numpy(), te.motor_id.to_numpy()

    # V03 preprocessing consistency
    m_tr, m_all = Xtr.mean(0), df[FEATURES].to_numpy(float).mean(0)
    e_tr = float(np.abs(bun["mean"] - m_tr).max() / np.abs(m_tr).max()); e_all = float(np.abs(bun["mean"] - m_all).max() / np.abs(m_all).max())
    s_tr = float(np.abs(bun["scale"] - Xtr.std(0, ddof=0)).max())
    scaler_ok = e_tr < 1e-8 and s_tr < 1e-8
    R.add("V03a", "Scaler learned from TRAINING motors only", PASS if scaler_ok and e_all > 1e-8 else FAIL,
          f"max relative mean difference vs training motors {e_tr:.2e}, vs all data {e_all:.2e}; max scale difference vs training {s_tr:.2e}. No imputation exists in the pipeline: inputs must be complete.")

    # V05 imbalance and leakage
    cnt = df.y.value_counts().sort_index(); xs = df[FEATURES].round(6).astype(str).agg("|".join, axis=1)
    cross = int(sum(set(xs[df.split == a]) & set(xs[df.split == b]) != set() for a, b in (("train", "test"), ("train", "val"), ("val", "test"))))
    dupcnt = len(set(xs[df.split == "train"]) & set(xs[df.split == "test"]))
    rng = np.random.default_rng(0); sc = []
    Xs = (Xtr - bun["mean"]) / bun["scale"]
    for _ in range(5):
        lr_ = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xs, rng.permutation(ytr))
        sc.append(f1_score(te.y, lr_.predict((Xte - bun["mean"]) / bun["scale"]), average="macro", labels=[0, 1, 2], zero_division=0))
    R.add("V05", "Class imbalance and leakage screens", WARN if np.mean(sc) > 0.45 or dupcnt else INFO,
          f"classes (Normal, Alert, Danger) overall {cnt.tolist()}, imbalance {cnt.max() / cnt.min():.1f}:1; test {np.bincount(yte, minlength=3).tolist()}. "
          f"Identical 14-value readings shared between train and test: {dupcnt}. Shuffled-label control on the test motors: macro-F1 {np.mean(sc):.3f} (should be near chance, ~0.33 or below). "
          f"Label-rule, family and time probes: run pm_data_audit.py (K03 to K05, P03).")

    # V06 reproduce the held-out evaluation from the SAVED model
    proba = forward(bun, Xte); M = compute_metrics(yte, proba); self_err = manual_selfcheck(yte, proba, M)
    rec = meta.get("test_metrics", {}).get("ANN")
    if rec is None:
        R.add("V06", "Reproduce the notebook's recorded test evaluation", NV, "training_metadata.json holds no test_metrics.ANN to compare with.")
        diffs = None
    else:
        cm_eq = rec["confusion_matrix"] == M["confusion_matrix"]
        keys = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mcc", "log_loss", "weighted_kappa_quadratic"]
        diffs = {k: abs(rec[k] - M[k]) for k in keys if k in rec}
        ok6 = cm_eq and max(diffs.values()) < 1e-5
        R.add("V06", "Reproduce the notebook's recorded test evaluation from the saved model", PASS if ok6 else FAIL,
              f"confusion matrix identical to the recorded one: {cm_eq}; largest metric difference {max(diffs.values()):.2e} (tolerance 1e-5, because Keras used float32 and this is float64). "
              + ("" if ok6 else "The saved weights, preprocessing, data or split are NOT the ones that produced the recorded numbers."))
    R.metrics["ANN_test"] = M
    R.add("V07a", "Metric implementation self-check", PASS if self_err < 1e-12 else FAIL, f"accuracy and per-class precision/recall recomputed by hand agree with scikit-learn to {self_err:.1e}")
    pred = proba.argmax(1)
    ci = {"macro_f1_95ci": cluster_ci(yte, pred, gte, lambda t, p: f1_score(t, p, labels=[0, 1, 2], average="macro", zero_division=0), B),
          "danger_recall_95ci": cluster_ci(yte, pred, gte, lambda t, p: recall_score(t == 2, p == 2, zero_division=0), B)}
    R.metrics["ANN_test_ci_by_motor"] = ci
    R.add("V07", "Test metrics (computed now from the saved model)", INFO,
          f"n={M['n']} from {len(set(gte))} motors | accuracy {M['accuracy']:.3f} | balanced accuracy {M['balanced_accuracy']:.3f} | macro-F1 {M['macro_f1']:.3f} (95% CI by motor {ci['macro_f1_95ci'][0]:.3f} to {ci['macro_f1_95ci'][1]:.3f}) | "
          f"Danger recall {M['per_class']['Danger']['recall']:.3f} (95% CI {ci['danger_recall_95ci'][0]:.3f} to {ci['danger_recall_95ci'][1]:.3f}) | MCC {M['mcc']:.3f} | Danger predicted as Normal: {M['danger_predicted_as_normal']}. "
          f"Few test motors: intervals are wide by construction.")

    # V08 imbalance-aware performance and Target B
    R.add("V08", "Precision-recall view for the imbalanced classes (Target A)", INFO,
          f"actual positives in test: { {c: M['per_class'][c]['support'] for c in CLASSES} }; average precision { {k: round(v, 3) for k, v in M['average_precision'].items()} } "
          f"versus prevalence (the score of a random ranker) { {k: round(v, 3) for k, v in M['prevalence'].items()} }")
    R.add("V08B", "Target B: verified failure within seven days", NV, target_b_gate(df, events))

    # V09 baselines under the same protocol
    bp = meta.get("baseline_params", {}); Xs_te = (Xte - bun["mean"]) / bun["scale"]
    base = {"Dummy (most frequent)": DummyClassifier(strategy="most_frequent").fit(Xs, ytr),
            "LogisticRegression": LogisticRegression(max_iter=3000, random_state=seed, **bp.get("logistic_regression", {})).fit(Xs, ytr),
            "RandomForest": RandomForestClassifier(random_state=seed, n_jobs=-1, **bp.get("random_forest", {})).fit(Xs, ytr)}
    rows, paired = {}, {}
    f1m = lambda t, p: f1_score(t, p, labels=[0, 1, 2], average="macro", zero_division=0)
    for n, mdl in base.items():
        pb = np.zeros((len(yte), 3)); pb[:, list(mdl.classes_)] = mdl.predict_proba(Xs_te); mb = compute_metrics(yte, pb); rows[n] = mb
        rng = np.random.default_rng(1); ms = np.unique(gte); idx = {g: np.where(gte == g)[0] for g in ms}; diff = []
        for _ in range(B):
            i = np.concatenate([idx[g] for g in rng.choice(ms, len(ms), replace=True)]); diff.append(f1m(yte[i], pred[i]) - f1m(yte[i], pb.argmax(1)[i]))
        paired[n] = {"macro_f1_ANN_minus_baseline": float(M["macro_f1"] - mb["macro_f1"]), "ci95": [float(x) for x in np.percentile(diff, [2.5, 97.5])], "share_of_resamples_ANN_better": float(np.mean(np.array(diff) > 0))}
    R.metrics["baselines_test"] = rows; R.metrics["paired_motor_bootstrap"] = paired
    recb = meta.get("test_metrics", {}); same = {n: (abs(recb[k]["macro_f1"] - rows[n]["macro_f1"]) < 1e-6) for n, k in (("RandomForest", "RandomForest"), ("LogisticRegression", "LogisticRegression")) if k in recb}
    R.add("V09", "ANN versus baselines, same training motors, same test motors", INFO,
          "; ".join(f"{n}: macro-F1 {r['macro_f1']:.3f}, Danger recall {r['per_class']['Danger']['recall']:.3f}, accuracy {r['accuracy']:.3f}" for n, r in rows.items())
          + f". ANN macro-F1 minus baseline with 95% CI by motor: { {n: (round(v['macro_f1_ANN_minus_baseline'], 3), [round(x, 3) for x in v['ci95']]) for n, v in paired.items()} }. "
          f"Baselines re-fitted with the notebook's recorded parameters and compared with its recorded baseline results: reproduced { same or 'not comparable' }. "
          f"If an interval contains 0, the data cannot separate the models.")

    # V10 saved model gives the same predictions as the original notebook
    chk = []
    ref_path = bd / "reference_inference.py"
    if ref_path.exists():
        try:
            spec = importlib.util.spec_from_file_location("ref_inf", ref_path); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            b2 = mod.load_bundle(bd, verify=True); p2 = mod.predict_proba_matrix(b2, Xte); chk.append(("bundle reference_inference.py vs this validator", float(np.abs(p2 - proba).max()), bool((p2.argmax(1) == pred).all())))
        except Exception as exc:  # noqa: BLE001 - a bundle that cannot be loaded and verified is a finding
            chk.append((f"bundle reference_inference.py could not load/verify: {type(exc).__name__}: {str(exc)[:100]}", float("inf"), False))
    else:
        chk.append(("bundle reference_inference.py", None, None))
    kp = bd / "model.keras"
    try:
        import keras
        if kp.exists():
            pk = keras.saving.load_model(kp).predict(((Xte - bun["mean"]) / bun["scale"]), verbose=0); chk.append(("model.keras (Keras) vs NumPy weights", float(np.abs(pk - proba).max()), bool((pk.argmax(1) == pred).all())))
        else:
            chk.append(("model.keras", None, None))
    except ImportError:
        chk.append(("model.keras (Keras not installed in this runtime)", None, None))
    if backend_dir:
        try:
            sys.path.insert(0, str(Path(backend_dir))); from app.services.artifacts import load_bundle as bl; from app.services.inference import forward as bf
            pbk = bf(bl(bd), Xte); chk.append(("backend inference code vs this validator", float(np.abs(pbk - proba).max()), bool((pbk.argmax(1) == pred).all())))
        except Exception as exc:  # noqa: BLE001
            chk.append((f"backend inference code ({type(exc).__name__}: {str(exc)[:80]})", None, None))
    tpf = Path(support) / "test_predictions.csv" if support else None
    if tpf is not None and tpf.exists():
        tp_ = pd.read_csv(tpf).set_index("record_id"); mine = pd.DataFrame(proba, index=te.record_id.values, columns=[f"p_{c}" for c in CLASSES])
        common = mine.index.intersection(tp_.index)
        if len(common) != len(te) or len(tp_) != len(te):
            chk.append((f"notebook test_predictions.csv covers {len(common)} of {len(te)} test rows (file has {len(tp_)})", float("inf"), False))
        else:
            dd = float(np.abs(mine.loc[common].to_numpy() - tp_.loc[common, [f"p_{c}" for c in CLASSES]].to_numpy()).max())
            chk.append((f"ROW-LEVEL: notebook's saved test predictions vs saved model ({len(common)} rows)", dd, bool((mine.loc[common].to_numpy().argmax(1) == tp_.loc[common, "y_pred"].to_numpy()).all())))
    ran = [c for c in chk if c[1] is not None]
    bad10 = [c for c in ran if c[1] > 1e-5 or not c[2]]
    R.add("V10", "Saved model reproduces the original notebook's predictions", FAIL if bad10 else (PASS if ran and diffs is not None and R.status("V06") == PASS else WARN if ran else NV),
          f"{[(n, None if d_ is None else f'{d_:.1e}', s) for n, d_, s in chk]}; recorded-confusion-matrix match: {R.status('V06')}. "
          + ("Row-level comparison with the original run was made (support files supplied)." if support and any(c[0].startswith("ROW-LEVEL") for c in chk) else
          "The notebook did not save per-row test predictions, so equality with the ORIGINAL run is shown only through the identical confusion matrix (V06). Add notebook_validation_support_cell.py to the training notebook and re-run to get row-level proof."))

    # V11 fresh runtime
    R.add("V11", "Artefacts load in a fresh interpreter with only NumPy", *fresh_load(bd, Xte[:5], proba[:5]))

    # V12 documentation
    R.add("V12", "Model version, schema, class mapping, seed, dataset version recorded", *document(meta, bun, sha, df, bd, support))

    # error analysis
    error_analysis(R, out, te, pred, proba, M)
    return finish(R, out, df, M, bun, sha, sha_in, data)


def target_b_gate(df, events):
    base = ("No Target B model exists in the bundle and the raw file has no failure-event column or event timestamps, so NO Target B performance can be computed or claimed. "
            "Condition labels (Danger) are not failures.")
    if not events:
        return base + " To start: supply a CSV of VERIFIED events with columns motor_id, event_time, event_type, source, verified."
    ev = pd.read_csv(events)
    need = {"motor_id", "event_time", "event_type", "source", "verified"}
    if not need <= set(ev.columns):
        return base + f" The events file lacks columns {sorted(need - set(ev.columns))}."
    syn = ev["source"].astype(str).str.contains("synthetic|simulated|demo", case=False).any()
    ev = ev[ev["verified"].astype(str).str.lower().isin(["yes", "true", "1", "y"])].copy()
    ev["event_time"] = pd.to_datetime(ev["event_time"], errors="coerce")
    pos = cens = 0
    last = df.groupby("motor_id").measured_at.max()
    for m, g in df.groupby("motor_id"):
        e = ev[ev.motor_id == m].event_time.dropna().to_numpy()
        for t in g.measured_at.to_numpy():
            if len(e) and ((e > t) & (e <= t + np.timedelta64(7, "D"))).any():
                pos += 1
            elif t + np.timedelta64(7, "D") > last[m].to_datetime64():
                cens += 1
    out = (f"Feasibility only: {len(ev)} verified events on {ev.motor_id.nunique()} motors; {pos} readings would be positive for 'event within 7 days'; {cens} readings lack a full 7-day window (censored); "
           f"{len(df) - pos - cens} negatives. No model B was evaluated. Positives per motor: {ev.groupby('motor_id').size().to_dict()}.")
    return ("SYNTHETIC EVENTS DETECTED: this is NOT industrial evidence. " if syn else "") + out


def fresh_load(bd, X5, p5):
    code = textwrap.dedent("""
        import sys, json, importlib.util, numpy as np
        d = sys.argv[1]; X = np.array(json.loads(sys.argv[2]))
        spec = importlib.util.spec_from_file_location('ref_inf', d + '/reference_inference.py'); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
        b = m.load_bundle(d, verify=True); print(json.dumps(m.predict_proba_matrix(b, X).tolist()))
        assert 'tensorflow' not in sys.modules and 'keras' not in sys.modules
    """)
    if not (Path(bd) / "reference_inference.py").exists():
        return NV, "reference_inference.py is not in the bundle."
    r = subprocess.run([sys.executable, "-I", "-c", code, str(Path(bd).resolve()), json.dumps(X5.tolist())], capture_output=True, text=True, timeout=120)
    if r.returncode != 0:
        return FAIL, "Fresh interpreter could not load and verify the bundle: " + r.stderr.strip()[-300:]
    got = np.array(json.loads(r.stdout.strip().splitlines()[-1])); diff = float(np.abs(got - p5).max())
    return (PASS if diff < 1e-9 else FAIL), (f"New python process (-I), NumPy only, no TensorFlow, no notebook state: manifest checksums verified, 5 test rows predicted, max difference from this run {diff:.1e}. "
                                             "For a stricter test also Runtime > Restart session in Colab and run only the validation notebook.")


def document(meta, bun, sha, df, bd, support=None):
    ds = meta.get("dataset", {}); facts = {
        "bundle_version": meta.get("bundle_version"), "created_utc": meta.get("created_utc"), "seed": meta.get("seed"), "task": meta.get("task"),
        "dataset_file": ds.get("file_name"), "dataset_sha256_recorded": ds.get("sha256"), "dataset_sha256_now": sha, "rows": ds.get("records"), "motors": ds.get("motors"),
        "time_range": ds.get("time_range"), "timestamp_format": ds.get("timestamp_format_assumed"), "features": bun["names"],
        "units": {f["name"]: f["unit"] for f in bun["schema"].get("features", [])}, "classes": dict(enumerate(bun["classes"])), "ann_config": meta.get("ann_config"),
        "provisional_label_provenance": meta.get("provisional_label_provenance"), "label_provenance": meta.get("label_provenance"),
        "libraries": meta.get("environment"), "test_evaluated_at_utc": meta.get("test_evaluated_at_utc"), "fast_debug_run": meta.get("fast_debug_run")}
    gaps = []
    if ds.get("sha256") != sha: gaps.append("recorded dataset SHA-256 differs from the file you supplied")
    if not (Path(bd) / "reproducibility_report.md").exists(): gaps.append("reproducibility_report.md missing")
    sup = Path(support) if support else None
    if not (isinstance(meta.get("split"), dict) and any(isinstance(v, list) for v in meta["split"].values())) and not (sup and (sup / "split_motors.json").exists()):
        gaps.append("split motor lists are not stored in a machine-readable file (only in reproducibility_report.md)")
    if not (sup and (sup / "test_predictions.csv").exists()): gaps.append("per-row test predictions were not saved")
    if not any(k in meta for k in ("notebook_sha256", "git_commit")): gaps.append("no notebook checksum or git commit is recorded")
    gaps.append("dataset has no human-readable version, only a SHA-256")
    for k in ("bundle_version", "seed", "created_utc", "environment"):
        if not meta.get(k): gaps.append(f"training_metadata.json lacks '{k}'")
    facts_missing = [k for k, v in facts.items() if v in (None, {}, "")]
    return (FAIL if any("differs" in g for g in gaps) else WARN if gaps or facts_missing else PASS), f"facts: {json.dumps(facts, default=str)} | gaps: {gaps} | empty fields: {facts_missing or 'none'}"


def error_analysis(R, out, te, pred, proba, M):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    t = te.assign(pred=pred, true=te.y, conf=proba.max(1)).copy(); t["ok"] = t.pred == t.true
    for k, c in enumerate(CLASSES): t[f"p_{c}"] = proba[:, k]
    t = t.sort_values(["motor_id", "measured_at"]); t["label_change_next_or_prev"] = t.groupby("motor_id").true.transform(lambda s: (s.diff().fillna(0) != 0) | (s.diff(-1).fillna(0) != 0))
    errs = t[~t.ok]
    pm = t.groupby("motor_id").agg(family=("family", "first"), n=("ok", "size"), errors=("ok", lambda s: int((~s).sum())), accuracy=("ok", "mean"), danger=("true", lambda s: int((s == 2).sum())))
    ordinal = {"adjacent (off by one class)": int(((errs.pred - errs.true).abs() == 1).sum()), "two classes apart (Normal<->Danger)": int(((errs.pred - errs.true).abs() == 2).sum())}
    near = {"errors at a reading next to a label change": int(errs.label_change_next_or_prev.sum()), "all errors": len(errs), "share of test readings next to a label change": float(t.label_change_next_or_prev.mean())}
    sens = [c for c in FEATURES if "velocity" in c or "temp" in c][:4]
    prof = t.groupby(["true", "ok"])[sens].median().round(3)
    R.tables["errors_by_motor"] = pm.reset_index().to_dict("records"); R.tables["ordinal_errors"] = ordinal; R.tables["errors_near_label_changes"] = near
    R.tables["median_sensor_values_by_truth_and_correctness"] = prof.reset_index().to_dict("records")
    R.tables["top_confusions"] = errs.groupby(["true", "pred"]).size().rename("count").reset_index().sort_values("count", ascending=False).to_dict("records")
    R.tables["most_confident_errors"] = errs.sort_values("conf", ascending=False).head(10)[["record_id", "motor_id", "measured_at", "true", "pred", "conf"]].astype(str).to_dict("records")
    keep = ["record_id", "motor_id", "family", "measured_at", "true", "pred", "conf"] + [f"p_{c}" for c in CLASSES] + sens
    errs[keep].to_csv(out / "test_errors.csv", index=False); t[keep].to_csv(out / "test_predictions.csv", index=False)
    cm = np.array(M["confusion_matrix"], float)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
    for a_, mat, ttl, norm in ((ax[0], cm, "Counts", False), (ax[1], cm / np.clip(cm.sum(1, keepdims=True), 1, None), "Row-normalised (recall)", True)):
        a_.imshow(mat, cmap="Blues", vmin=0, vmax=1 if norm else mat.max()); a_.set_xticks(range(3), CLASSES); a_.set_yticks(range(3), CLASSES); a_.set_xlabel("predicted"); a_.set_ylabel("actual"); a_.set_title(ttl)
        for i in range(3):
            for j in range(3): a_.text(j, i, f"{mat[i, j]:.2f}" if norm else int(mat[i, j]), ha="center", va="center", color="white" if mat[i, j] > (0.5 if norm else mat.max() / 2) else "black")
    fig.tight_layout(); fig.savefig(out / "confusion_matrix.png", dpi=150); plt.close(fig); R.figures.append("confusion_matrix.png")
    from sklearn.metrics import precision_recall_curve
    fig, ax = plt.subplots(figsize=(5, 3.8))
    for k, c in enumerate(CLASSES):
        if (te.y == k).any():
            p_, r_, _ = precision_recall_curve((te.y == k).astype(int), proba[:, k]); ax.plot(r_, p_, label=f"{c} (AP {M['average_precision'][c]:.2f}, prevalence {M['prevalence'][c]:.2f})")
    ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02); ax.legend(fontsize=7); ax.set_title("Precision-recall by class (test motors)"); fig.tight_layout(); fig.savefig(out / "precision_recall.png", dpi=150); plt.close(fig); R.figures.append("precision_recall.png")
    fig, ax = plt.subplots(figsize=(6, 3.2)); ax.bar(pm.index, pm.accuracy); ax.set_ylim(0, 1); ax.set_ylabel("accuracy"); ax.set_title("Accuracy per test motor"); fig.tight_layout(); fig.savefig(out / "per_motor_accuracy.png", dpi=150); plt.close(fig); R.figures.append("per_motor_accuracy.png")
    R.add("V13", "Error analysis", INFO, f"{len(errs)} errors in {len(t)} test readings. {ordinal}. {near}. Lowest-accuracy motors: {pm.sort_values('accuracy').head(3)[['n', 'errors', 'accuracy']].round(3).to_dict('index')}. Files: test_errors.csv, test_predictions.csv (contain record and motor IDs: keep them private).")


def finish(R, out, df, M, bun, sha, sha_in, data):
    R.add("V99", "Inputs unchanged by this validation", PASS if sha256_of(data) == sha_in else FAIL, f"raw file SHA-256 before = after = {sha_in[:16]}...")
    prov = (bun["meta"].get("provisional_label_provenance"))
    verdict = ("NOT VALIDATED: " + ", ".join(c["id"] for c in R.checks if c["status"] == FAIL)) if any(c["status"] == FAIL for c in R.checks) else \
        "Target A: reproduced from the saved artefacts. Label provenance " + ("UNDOCUMENTED (provisional)" if prov else "is MARKED documented (not verifiable by a program)") + ". Target B: NOT VALIDATED (no verified failure events, no model)."
    meta = {"validation_version": VALIDATION_VERSION, "run_at": datetime.now().isoformat(timespec="seconds"), "verdict": verdict, "data_sha256": sha_in}
    (out / "validation_metrics.json").write_text(json.dumps({**meta, "checks": R.checks, "metrics": R.metrics, "tables": R.tables}, indent=2, default=str), encoding="utf-8")
    L = ["# Model validation report (generated from the files you supplied)", f"- Run {meta['run_at']} | validator v{VALIDATION_VERSION} | raw file SHA-256 `{sha_in}`", f"- **Verdict: {verdict}**", "",
         "| ID | Status | Check | Detail |", "|---|---|---|---|"] + [f"| {c['id']} | **{c['status']}** | {c['title']} | {str(c['detail']).replace('|', '/')} |" for c in R.checks]
    for k, v in R.tables.items(): L += ["", f"### {k}", "```", json.dumps(v, indent=1, default=str)[:3500], "```"]
    if M:
        L += ["", "### Test metrics (Target A)", "```", json.dumps({k: M[k] for k in M if k not in ("confusion_matrix",)}, indent=1)[:3000], "```", f"Confusion matrix (rows actual, columns predicted: {CLASSES}): `{M['confusion_matrix']}`"]
    L += [""] + [f"![{f}]({f})" for f in R.figures]
    (out / "validation_report.md").write_text("\n".join(L), encoding="utf-8")
    return R, meta


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True); ap.add_argument("--bundle", required=True); ap.add_argument("--out", default="validation_outputs")
    ap.add_argument("--events"); ap.add_argument("--backend-dir"); ap.add_argument("--support", help="folder written by notebook_validation_support_cell.py (split_motors.json, test_predictions.csv)"); ap.add_argument("--seed", type=int, default=42); ap.add_argument("--bootstrap", type=int, default=2000)
    a = ap.parse_args(argv)
    if Path(a.out).resolve() in (Path(a.data).resolve().parent, Path(a.bundle).resolve()):
        print("Choose an --out folder that is not the data folder or the bundle."); return 2
    try:
        R, meta = run(a.data, a.bundle, a.out, a.events, a.backend_dir, a.seed, a.bootstrap, a.support)
    except (ValueError, FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        print("Could not run:", type(exc).__name__, exc); return 2
    for c in R.checks: print(f"[{c['status']:13}] {c['id']:5} {c['title']}")
    print("\n" + meta["verdict"]); print("Outputs:", Path(a.out).resolve())
    return 1 if any(c["status"] == FAIL for c in R.checks) else 0


if __name__ == "__main__":
    sys.exit(main())


## Step 2. Provide your files
1. The **original** `motor_predictive_maintenance_master.csv`.
2. The model bundle: either the folder `pm_condition_ann_v1.0.0` already in `/content/outputs`, or the **zip** your training notebook downloaded (`pm_condition_ann_v1.0.0.zip`).
3. Optional but recommended: the `validation_support` folder (made by the extra cell in the training notebook) for row-level proof. Upload it as a zip named `validation_support.zip`.

In [ ]:
import os, glob, zipfile, hashlib
def pick(env, default):
    p = os.environ.get(env, default)
    return p if os.path.exists(p) else None

DATA = pick("PM_DATA_PATH", "/content/motor_predictive_maintenance_master.csv")
BUNDLE = os.environ.get("PM_BUNDLE_PATH") or (sorted(glob.glob("/content/outputs/pm_condition_ann_v*/"))[-1:] or [None])[0]
SUPPORT = os.environ.get("PM_SUPPORT_PATH") or ("/content/outputs/validation_support" if os.path.isdir("/content/outputs/validation_support") else None)
if not (DATA and BUNDLE):
    from google.colab import files
    print("Choose: the raw CSV, the bundle zip (pm_condition_ann_*.zip) and optionally validation_support.zip")
    for name in files.upload():
        if name.endswith(".csv"): DATA = "/content/" + name
        elif name.startswith("validation_support"):
            zipfile.ZipFile("/content/" + name).extractall("/content/validation_support_x"); SUPPORT = (glob.glob("/content/validation_support_x/**/test_predictions.csv", recursive=True) or [None])[0]
            SUPPORT = os.path.dirname(SUPPORT) if SUPPORT else None
        elif name.endswith(".zip"):
            zipfile.ZipFile("/content/" + name).extractall("/content/bundle_x"); m = glob.glob("/content/bundle_x/**/MANIFEST.json", recursive=True); BUNDLE = os.path.dirname(m[0]) if m else None
print("data   :", DATA, "\nbundle :", BUNDLE, "\nsupport:", SUPPORT)
assert DATA and BUNDLE, "Raw CSV and bundle are both required."
print("SHA-256 of the original CSV:", hashlib.sha256(open(DATA, "rb").read()).hexdigest())

## Step 3. Run the validation
About 30 to 90 seconds. Read each line: **PASS** = checked and held; **FAIL** = the saved artefacts do not match what was recorded; **WARN** = a documented gap; **INFO** = a result to read; **NOT VALIDATED** = it could not be checked, with the reason in the report.

In [ ]:
cmd = f'python pm_model_validation.py --data "{DATA}" --bundle "{BUNDLE}" --out /content/validation_outputs'
if SUPPORT: cmd += f' --support "{SUPPORT}"'
print(cmd, "\n")
!{cmd}

## Step 4. Read the results
The full report is `validation_outputs/validation_report.md`. Below: the checks that are not PASS or INFO, then the figures. Metrics are computed from your files only.

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
rep = json.load(open("/content/validation_outputs/validation_metrics.json"))
print("VERDICT:", rep["verdict"])
pd.set_option("display.max_colwidth", 600, "display.width", 250)
chk = pd.DataFrame(rep["checks"])
display(chk[~chk.status.isin(["PASS", "INFO"])][["id", "status", "title", "detail"]])
m = rep["metrics"].get("ANN_test")
if m:
    print("\nTEST SET (ANN):", {k: round(v, 3) for k, v in m.items() if isinstance(v, float)})
    print("Confusion matrix (rows actual, columns predicted: Normal, Alert, Danger):", m["confusion_matrix"])
    display(pd.DataFrame(m["per_class"]).T.round(3))
    print("95% intervals by motor:", rep["metrics"]["ANN_test_ci_by_motor"])
    display(pd.DataFrame(rep["metrics"]["baselines_test"]).T[["accuracy", "balanced_accuracy", "macro_f1", "mcc"]].round(3))
    display(pd.DataFrame(rep["metrics"]["paired_motor_bootstrap"]).T.round(3))
for f in ["confusion_matrix.png", "precision_recall.png", "per_motor_accuracy.png"]:
    p = "/content/validation_outputs/" + f
    if os.path.exists(p): display(Image(p))

## Step 5. Error analysis
Which readings and motors the model gets wrong, and whether errors sit next to label changes.

In [ ]:
for k in ["errors_by_motor", "top_confusions", "ordinal_errors", "errors_near_label_changes", "most_confident_errors"]:
    t = rep["tables"].get(k)
    if t is not None:
        print("\n##", k); display(pd.DataFrame(t) if isinstance(t, list) else pd.Series(t))

## Step 6 (optional). Target B feasibility
Only if you obtain a CSV of **verified** failure events with columns `motor_id, event_time, event_type, source, verified`. This reports how many positive events exist. It never scores a model, because none exists.

In [ ]:
EVENTS = "/content/verified_failure_events.csv"   # upload it to /content first
if os.path.exists(EVENTS):
    !python pm_model_validation.py --data "{DATA}" --bundle "{BUNDLE}" --out /content/validation_outputs_b --events "{EVENTS}" | grep V08B
else:
    print("No events file: Target B stays NOT VALIDATED (this is the correct result for your current data).")

## Step 7. Save the outputs
`test_errors.csv` and `test_predictions.csv` contain record and motor IDs: keep them private.

In [ ]:
!cd /content && zip -qr validation_outputs.zip validation_outputs
try:
    from google.colab import files; files.download("/content/validation_outputs.zip")
except ImportError:
    print("Outputs are in /content/validation_outputs")